# Cora training with the Neo4j remote backend

This notebook trains a 2-layer GraphSAGE classifier on **Cora** (consisting of ~2.7k nodes) using PyG's `DatabaseFeatureStore` / `DatabaseGraphStore` / `DatabaseSampler` abstractions, backed by a running **Neo4j** database.

Concrete implementations exercised here:

* `Neo4jGraphStore` (`examples/neo4j/data/neo4j_graph_store.py`) — runs Cypher queries and returns raw (non-decoded) result.
* `Neo4jFeatureStore` (`examples/neo4j/data/neo4j_feature_store.py`) — fetches node properties (`x`, `y`) per mini-batch, with an optional `LRUFeatureCache`, and decodes the result.
* `Neo4jGraphSAGESampler` (`examples/neo4j/neo4j_samplers.py`) — Defines multi-hop neighbor sampling query in Cypher that pushes BFS expansion into Neo4j (APOC required). The sampler also decodes the result.

## Tips for making Neo4j-backed GNN training faster

1. **Create an index on the node id property.**  
   This makes seed-node lookup and feature lookup fast [(Search-performance indexes)](https://neo4j.com/docs/cypher-manual/current/indexes/search-performance-indexes/).

2. **Store features as compact `byte[]` values.**  
   This keeps feature data in a contiguous binary format, reducing driver decoding overhead and making conversion into tensors much cheaper.

3. **Use Java user-defined procedures (UDPs).**  
   Sampling and feature fetching are performance-critical and can be faster as imperative procedures than as general Cypher queries. For example, a feature-fetching procedure can return one compact byte buffer for the whole batch instead of one record per node [(Neo4j Java User-defined procedures)](https://neo4j.com/docs/java-reference/current/extending-neo4j/procedures/).

   The UDPs are called through cypher queries like:

    ```cypher
    CALL gnnProcedures.sampling.graphSAGE( 
            $seed_ids, 
            $fanout,
        )
    ```

    > Note: UDPs can also be used to implement low-latency inference in Neo4j, where the full forward pass is executed in Neo4j


4. **Avoid Cypher queries that materialize full neighborhoods before sampling.**  
   Patterns such as `collect(r)` followed by `apoc.coll.randomItems(...)` first load all candidate relationships into memory, even when only a small fanout is needed. This can be especially costly for high-degree nodes.

5. **Use the Rust extension for the Neo4j Python driver.**  
   The `neo4j-rust-ext` package keeps the same Python API as the regular driver, but speeds up Bolt deserialization. Neo4j’s Python driver documentation reports a 3x to 10x speedup compared with the regular driver [(Neo4j Developer Blog, “Neo4j Python Driver 10x Faster With Rust”)](https://neo4j.com/blog/developer/python-driver-10x-faster-with-rust/).

## Prerequisites

1. A running Neo4j instance (5.x) reachable over Bolt. The [APOC plugin](https://neo4j.com/docs/apoc/).
2. Python dependencies: `torch`, `torch_geometric` and `neo4j`.

Run all cells top to bottom. The **Prepare data** section (after the connection cell) downloads Cora via PyG and ingests it into Neo4j — no separate script needed. Set `WIPE = True` there to clear any existing graph first.

In [1]:
!pip install -q neo4j


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import sys
from pathlib import Path

# Make the example modules importable when running this notebook from
# `examples/neo4j/example_with_cora/`.
HERE = Path.cwd()
NEO4J_DIR = HERE.parent  # examples/neo4j
for p in (str(NEO4J_DIR), str(NEO4J_DIR / 'data')):
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
import torch.nn.functional as F
from neo4j import GraphDatabase

from torch_geometric.data.database_feature_store import LRUFeatureCache
from torch_geometric.loader import NodeLoader
from torch_geometric.nn import SAGEConv

from neo4j_feature_store import Neo4jFeatureStore
from neo4j_graph_store import Neo4jGraphStore
import importlib
import neo4j_samplers
importlib.reload(neo4j_samplers)
from neo4j_samplers import Neo4jGraphSAGESampler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

/Users/victorpekkari/Desktop/thesis-and-pyg/pytorch_geometric/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## Connection settings

Edit the four values below to point at your Neo4j instance. The values shown are placeholders for a local Bolt server with the default `neo4j` database.

**Do not commit real credentials.** For anything beyond a local demo, read them from the environment instead of hard-coding them, e.g.:

```python
import os
URI      = os.environ["NEO4J_URI"]
USER     = os.environ["NEO4J_USER"]
PASSWORD = os.environ["NEO4J_PASSWORD"]
DATABASE = os.environ.get("NEO4J_DATABASE", "neo4j")
```

In [3]:
URI         = "bolt://localhost:7687"
USER        = "neo4j"
PASSWORD    = "testing_pyg123"
DATABASE    = "neo4j"
NODE_LABEL  = "Paper"   # change to match your node label
NODEID_PROP = "nodeId"  # change to match your node id property


## Prepare data

Cora is a graph consisting of papers as nodes, and citations as edges. The cell below downloads Cora via PyG and ingests it into Neo4j as `(:Paper)-[:CITES]->(:Paper)`. The implementation is in the collapsed cell below; set `wipe=True` in the call cell to clear the database first.

The ingest uses batched `UNWIND` Cypher via the Python driver. For a graph this size (~2.7k nodes, ~10k edges) it finishes in well under a second and keeps the notebook self-contained. For much larger datasets prefer `neo4j-admin database import` (offline bulk loader, requires the DB stopped) or `apoc.periodic.iterate` for online batched writes; the choice here is optimized for clarity, not throughput.

Labels and property names are parametrized — change `node_label`, `rel_type`, `nodeid_prop` in the call below if you want to mirror your own dataset schema.

In [4]:
import re
from pathlib import Path

from neo4j import GraphDatabase

from torch_geometric.datasets import Planetoid

_NODE_BATCH = 500
_EDGE_BATCH = 5000
_IDENT_RE = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def _check_ident(name, value):
    if not _IDENT_RE.match(value):
        raise ValueError(f"{name}={value!r} is not a valid Cypher identifier")


def _ingest_cora(uri, user, pwd, database, *, wipe=False,
                 planetoid_root="./data/Planetoid",
                 node_label="Paper", rel_type="CITES",
                 nodeid_prop="nodeId"):
    # `node_label`, `rel_type`, `nodeid_prop` are interpolated into the
    # Cypher strings below. They cannot be passed as query parameters in
    # Cypher, so we validate them as plain identifiers to avoid injection.
    for name, val in (("node_label", node_label), ("rel_type", rel_type),
                      ("nodeid_prop", nodeid_prop)):
        _check_ident(name, val)

    def _chunks(seq, n):
        for i in range(0, len(seq), n):
            yield seq[i:i + n]

    root = Path(planetoid_root).expanduser().resolve()
    print(f"Loading Cora from {root} ...")
    d = Planetoid(root=str(root), name="Cora")[0]
    print(f"  nodes={d.num_nodes}, edges={d.edge_index.shape[1]}, "
          f"dim={d.x.shape[1]}, classes={int(d.y.max()) + 1}")

    nodes = [{nodeid_prop: i, "x": bytes(d.x[i].numpy().tobytes()),
              "y": int(d.y[i])} for i in range(d.num_nodes)]
    edges = [{"src": int(d.edge_index[0, j]), "dst": int(d.edge_index[1, j])}
             for j in range(d.edge_index.shape[1])]

    with GraphDatabase.driver(uri, auth=(user, pwd)) as drv, \
            drv.session(database=database) as s:
        if wipe:
            print("Wiping existing graph ...")
            s.run("MATCH (n) DETACH DELETE n")
        s.run(f"CREATE CONSTRAINT {node_label.lower()}_{nodeid_prop} "
              f"IF NOT EXISTS FOR (p:{node_label}) "
              f"REQUIRE p.{nodeid_prop} IS UNIQUE")
        print(f"Inserting {len(nodes)} nodes ...")
        for batch in _chunks(nodes, _NODE_BATCH):
            s.run(f"UNWIND $rows AS r "
                  f"MERGE (p:{node_label} {{{nodeid_prop}: r.{nodeid_prop}}}) "
                  f"SET p.x = r.x, p.y = r.y",
                  rows=batch)
        print(f"Inserting {len(edges)} edges ...")
        for batch in _chunks(edges, _EDGE_BATCH):
            s.run(f"UNWIND $rows AS r "
                  f"MATCH (a:{node_label} {{{nodeid_prop}: r.src}}) "
                  f"MATCH (b:{node_label} {{{nodeid_prop}: r.dst}}) "
                  f"MERGE (a)-[:{rel_type}]->(b)", rows=batch)
        c = s.run(f"MATCH (p:{node_label}) WITH count(p) AS n "
                  f"MATCH (:{node_label})-[c:{rel_type}]->(:{node_label}) "
                  f"RETURN n, count(c) AS e").single()
        print(f"Done \u2014 {c['n']} :{node_label} nodes, "
              f"{c['e']} :{rel_type} edges.")

    import torch
    train_idx = d.train_mask.nonzero(as_tuple=False).view(-1)
    val_idx   = d.val_mask.nonzero(as_tuple=False).view(-1)
    test_idx  = d.test_mask.nonzero(as_tuple=False).view(-1)
    return train_idx, val_idx, test_idx

In [5]:
train_idx, val_idx, test_idx = _ingest_cora(
    URI, USER, PASSWORD, DATABASE,
    wipe=True,
    node_label=NODE_LABEL,
    nodeid_prop=NODEID_PROP,
)
print(f'train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}')


Loading Cora from /Users/victorpekkari/Desktop/thesis-and-pyg/pytorch_geometric/examples/neo4j/example_with_cora/data/Planetoid ...
  nodes=2708, edges=10556, dim=1433, classes=7
Wiping existing graph ...
Inserting 2708 nodes ...
Inserting 10556 edges ...
Done — 2708 :Paper nodes, 10556 :CITES edges.
train=140  val=500  test=1000


## Create an index for fast lookups

> **Important:** Create an index on the node ID property before sampling.
> Without it, seed-node lookups and neighbor expansion become much slower.

In [6]:
with GraphDatabase.driver(URI, auth=(USER, PASSWORD)) as _drv, \
        _drv.session(database=DATABASE) as s:
    idx_name = f"{NODE_LABEL.lower()}_{NODEID_PROP}_idx"
    s.run(f"CREATE INDEX {idx_name} IF NOT EXISTS "
          f"FOR (p:{NODE_LABEL}) ON (p.{NODEID_PROP})")
    print("Index created (or already existed).")

Index created (or already existed).


## Build the graph store

`Neo4jGraphStore` is a concrete `DatabaseGraphStore`. It owns a lazy Bolt driver and exposes a single `query_db(query, params)` hook that runs a Cypher query and returns the raw record. Decoding the record into `(node, row, col)` COO tensors lives in the sampler (`Neo4jGraphSAGESampler._decode_node_sampling_record`), so the same `query_db` channel is reusable for sampling, edge lookups, and ad-hoc inspection.

Sampling is pushed into Cypher; the graph store itself does not materialize the global edge index.

In [7]:
graph_store = Neo4jGraphStore(
    uri=URI,
    user=USER,
    pwd=PASSWORD,
    database_name=DATABASE,
    nodeid_property='nodeId',
)

assert graph_store.apoc_available(), (
    'APOC is required by Neo4jGraphSAGESampler. Install the APOC plugin '
    'in your Neo4j instance and restart it.'
)

## Build the feature store

`Neo4jFeatureStore` is a concrete `DatabaseFeatureStore`. The `attr_map` declares which Neo4j node properties back which PyG tensor attributes, here we register `x` (byte[] array) and `y` (int64 label). `x` can also be a float arrar but that makes the data processing in the neo4j a lot slower.

Caching is pluggable via the `cache` argument; `LRUFeatureCache` is the default in-memory bounded cache. Pass `cache=None` to disable. The `LRUFeatureCache` will get replicated across sampling workers, to use one common cache for all sampling workers, implement for example a redis cache.

In [8]:
attr_map = {
    # 'x' is stored as a raw byte array in Neo4j (numpy float32 tobytes()).
    # encoding='byte[]' tells the feature store to deserialize it back to float32.
    # Storing as bytes is much faster to process in Neo4j than a list of floats.
    'x': {'property': 'x', 'dtype': 'float32', 'encoding': 'byte[]'},
    # 'y' is a plain integer property — no encoding needed.
    'y': {'property': 'y', 'dtype': 'int64'},
}

feature_store = Neo4jFeatureStore(
    attr_map=attr_map,
    uri=URI,
    user=USER,
    pwd=PASSWORD,
    database_name=DATABASE,
    nodeid_property='nodeId',
    default_node_label='Paper',
    cache=LRUFeatureCache(maxsize=4096),
)

## Build the sampler

`Neo4jGraphSAGESampler` is a concrete `DatabaseSampler`. Its `_build_node_sampling_query` compiles a single Cypher query at construction time that performs `len(num_neighbors)`-hop BFS sampling — with each hop expanding from the current frontier, picking up to `k` neighbors uniformly at random (or all when `k = -1`).

In [9]:
sampler = Neo4jGraphSAGESampler(
    graph_store=graph_store,
    num_neighbors=[20, 10],
    direction='incoming',
    # not needed in homogeneous graphs
    # node_label='Paper',
    # rel_type='CITES',
)

## Build NodeLoaders

PyG's `NodeLoader` accepts a `(feature_store, graph_store)` tuple in place of an in-memory `Data` object. It hands seed IDs to the sampler, then queries the feature store for the resulting subgraph nodes.

In [10]:
def make_loader(input_nodes, batch_size, shuffle):
    return NodeLoader(
        data=(feature_store, graph_store),
        node_sampler=sampler,
        input_nodes=input_nodes,
        batch_size=batch_size,
        shuffle=shuffle,
    )

train_loader = make_loader(train_idx, batch_size=32, shuffle=True)
val_loader   = make_loader(val_idx,   batch_size=64, shuffle=False)
test_loader  = make_loader(test_idx,  batch_size=64, shuffle=False)

## Define the model

Standard 2-layer `SAGEConv` stack — nothing Neo4j-specific. The mini-batches arrive in PyG's normal `Data` shape (`x`, `edge_index`, `y`), with seed nodes at the front of the local graph.

In [11]:
FEATURE_DIM = 1433
NUM_CLASSES = 7

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)

model = GraphSAGE(FEATURE_DIM, hidden_dim=30, out_dim=NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=5e-4)

## Train loop

In [12]:
NUM_EPOCHS = 10

def train_one_epoch():
    model.train()
    total_loss = total = 0
    for batch in train_loader:
        batch = batch.to(device)
        n_seeds = batch.batch_size
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)[:n_seeds]
        y = batch.y[:n_seeds]
        loss = F.cross_entropy(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * n_seeds
        total += n_seeds
    return total_loss / max(total, 1)

@torch.no_grad()
def accuracy(loader):
    model.eval()
    correct = total = 0
    for batch in loader:
        batch = batch.to(device)
        n_seeds = batch.batch_size
        pred = model(batch.x, batch.edge_index)[:n_seeds].argmax(dim=-1)
        correct += int((pred == batch.y[:n_seeds]).sum())
        total += n_seeds
    return correct / max(total, 1)

for epoch in range(1, NUM_EPOCHS + 1):
    loss = train_one_epoch()
    val_acc = accuracy(val_loader)
    print(f'Epoch {epoch:02d} | loss={loss:.4f} | val_acc={val_acc:.4f}')

Epoch 01 | loss=1.8676 | val_acc=0.6280
Epoch 02 | loss=1.1144 | val_acc=0.7420
Epoch 03 | loss=0.4764 | val_acc=0.7700
Epoch 04 | loss=0.2151 | val_acc=0.7580
Epoch 05 | loss=0.1022 | val_acc=0.7580
Epoch 06 | loss=0.0601 | val_acc=0.7620
Epoch 07 | loss=0.0334 | val_acc=0.7600
Epoch 08 | loss=0.0185 | val_acc=0.7560
Epoch 09 | loss=0.0104 | val_acc=0.7600
Epoch 10 | loss=0.0167 | val_acc=0.7540


## Test accuracy

In [13]:
print(f'Test accuracy: {accuracy(test_loader):.4f}')

Test accuracy: 0.7880


## Cleanup

Close the lazy drivers held by the stores and the explicit driver used to fetch split indices.

In [14]:
feature_store.close()
graph_store.close()